In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
matches_df = spark.table("ipl_analytics.bronze.matches_raw")

In [0]:
display(matches_df)

### Check Total Records


In [0]:
print("total record :",matches_df.count())

### Check Duplicate Records

In [0]:
duplicates = (
    matches_df.groupBy("match_id")
              .count()
              .filter(col("count") > 1)
)

display(duplicates)

### Remove Duplicates

In [0]:
matches_clean_df = matches_df.dropDuplicates(["match_id"])

### Check Null Values


In [0]:
print("match_id nulls:", matches_clean_df.filter(col("match_id").isNull()).count())

print("season nulls:", matches_clean_df.filter(col("season").isNull()).count())

print("match_date nulls:", matches_clean_df.filter(col("match_date").isNull()).count())

print("venue nulls:", matches_clean_df.filter(col("venue").isNull()).count())

print("team1 nulls:", matches_clean_df.filter(col("team1").isNull()).count())

print("team2 nulls:", matches_clean_df.filter(col("team2").isNull()).count())

print("winner nulls:", matches_clean_df.filter(col("winner").isNull()).count())

### Data Type Verification

In [0]:
matches_clean_df.printSchema()

## Business Rule Validation

### Check team1 vs team2

In [0]:
matches_clean_df.filter(col("team1") == col("team2")).show()


## Validate winner

In [0]:
matches_clean_df = matches_clean_df.filter(
    (col("winner") == col("team1")) |
    (col("winner") == col("team2"))
)

### toss_winner

In [0]:
matches_clean_df = matches_clean_df.filter(
    (col("toss_winner") == col("team1")) |
    (col("toss_winner") == col("team2"))
)

## Write Silver Table

In [0]:
(matches_clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ipl_analytics.silver.matches"))

## Verify Silver Table

In [0]:
display(spark.table("ipl_analytics.silver.matches"))

## SQL Verification

In [0]:
%sql
select count(*) from ipl_analytics.silver.matches;

# --------------------------------------------
# Players Silver Transformation
# --------------------------------------------

### Read Bronze Players Table

In [0]:
players_df = spark.table("ipl_analytics.bronze.players_raw")

display(players_df)

### Check Total Records

In [0]:
print("Total Records :", players_df.count())

### Check Duplicate Player IDs

In [0]:
players_df.groupBy("player_id") \
          .count() \
          .filter(col("count") > 1) \
          .show()

In [0]:
players_clean_df = players_df.dropDuplicates(["player_id"])

In [0]:
players_clean_df.groupBy("player_id") \
          .count() \
          .filter(col("count") > 1) \
          .show()

### Remove Duplicates

In [0]:
players_clean_df = players_df.dropDuplicates(["player_id"])

### Check Null Values

In [0]:
print("player_id nulls :", players_clean_df.filter(col("player_id").isNull()).count())

print("player_name nulls :", players_clean_df.filter(col("player_name").isNull()).count())

print("role nulls :", players_clean_df.filter(col("role").isNull()).count())

print("team nulls :", players_clean_df.filter(col("team").isNull()).count())

### Remove NULL Player Names

In [0]:
players_clean_df = players_clean_df.filter(
    col("player_name").isNotNull()
)

In [0]:
players_clean_df = players_clean_df.filter(
    col("player_id").isNotNull()
)

###  Remove Blank Player Names

In [0]:
players_clean_df = players_clean_df.filter(
    trim(col("player_name")) != ""
)

### Verify Data Types

In [0]:
players_clean_df.printSchema()

### Business Rule 1
Role should be only:

- Batsman
- Bowler
- All-rounder
- Wicketkeeper

In [0]:
players_clean_df.select("role").distinct().show()


In [0]:
players_clean_df.select("role").distinct().show()

### Player name should not be empty.

In [0]:
players_clean_df.filter(trim(col("player_name")) == "").show()

### Every team in Players must exist in Teams.

In [0]:
teams_df=spark.table("ipl_analytics.bronze.teams_raw")

### Remove Invalid Teams

In [0]:
valid_teams = [
    "Delhi Capitals",
    "Rajasthan Royals",
    "Mumbai Indians",
    "Kolkata Knight Riders",
    "Royal Challengers Bengaluru",
    "Punjab Kings",
    "Chennai Super Kings",
    "Gujarat Titans",
    "Lucknow Super Giants",
    "Sunrisers Hyderabad"
]

players_clean_df = players_clean_df.filter(
    col("team").isin(valid_teams)
)


### Remove Invalid Roles

In [0]:
valid_roles = [
    "Batsman",
    "Bowler",
    "All-rounder",
    "Wicketkeeper"
]

players_clean_df = players_clean_df.filter(
    col("role").isin(valid_roles)
)

### Standardize Text


In [0]:
from pyspark.sql.functions import initcap, trim

players_clean_df = (
    players_clean_df
    .withColumn("player_name", initcap(trim(col("player_name"))))
    .withColumn("team", upper(trim(col("team"))))
)

### Write Silver Table

In [0]:
players_clean_df.write .format("delta").mode("overwrite").saveAsTable("ipl_analytics.silver.players")

### Verification

In [0]:
display(spark.table("ipl_analytics.silver.players"))

### SQL Verification

In [0]:
%sql
select count(*) from ipl_analytics.silver.players;

# --------------------------------------------
# Teams Silver Transformation
# --------------------------------------------


In [0]:
teams_df = spark.table("ipl_analytics.bronze.teams_raw")

display(teams_df)

In [0]:
teams_df.printSchema()

In [0]:
print("Total Records :", teams_df.count())

### Duplicate Check

In [0]:
from pyspark.sql.functions import count

duplicate_rows = (
    teams_df
    .groupBy(*teams_df.columns)
    .count()
    .filter(col("count") > 1)
)

display(duplicate_rows)

In [0]:
teams_clean_df = teams_df.dropDuplicates(["team_id"])

### Null Check

In [0]:
print("team_id nulls :", teams_df.filter(col("team_id").isNull()).count())
print("team_name nulls :", teams_df.filter(col("team_name").isNull()).count())
print("coach nulls :", teams_df.filter(col("coach").isNull()).count())
print("captain nulls :", teams_df.filter(col("captain").isNull()).count())
print("home_ground nulls :", teams_df.filter(col("home_ground").isNull()).count())

### Handle NULL Values

In [0]:
from pyspark.sql.functions import col, trim

# Option A: Filter out rows where coach is 'Unknown', null, or empty
teams_clean_df = teams_clean_df.filter(
    (col("coach").isNotNull()) & 
    (col("coach") != "Unknown") & 
    (trim(col("coach")) != "")
)

display(teams_clean_df)

### Business Rule 1

In [0]:
teams_df.filter(trim(col("team_name")) == "").show()

### Remove Leading/Trailing Spaces


In [0]:
from pyspark.sql.functions import *

teams_clean_df = (
    teams_clean_df
    .withColumn("team_name", trim(col("team_name")))
    .withColumn("coach", trim(col("coach")))
    .withColumn("home_ground", trim(col("home_ground")))
    .withColumn("captain", trim(col("captain")))
)

### Business Rule 2

In [0]:
teams_df.filter(~col("team_id").startswith("T")).show()

### Standardization

In [0]:
teams_clean_df = (
    teams_clean_df
    .withColumn("team_name", upper(col("team_name")))
    .withColumn("coach", initcap(col("coach")))
    .withColumn("home_ground", initcap(col("home_ground")))
    .withColumn("captain", initcap(col("captain")))
)

### Remove Invalid Team

In [0]:
teams_clean_df = teams_clean_df.filter(
    col("team_name") != "INVALID XI"
)

### Write Silver

In [0]:
teams_clean_df.printSchema()

In [0]:
spark.table("ipl_analytics.silver.teams").printSchema()

In [0]:
%sql
DROP TABLE IF EXISTS ipl_analytics.silver.teams;

In [0]:
(teams_clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ipl_analytics.silver.teams"))

# --------------------------------------------
# Deliveries (Ball-by-Ball) Transformation
# 

In [0]:
deliveries_df = spark.table("ipl_analytics.bronze.ball_by_ball_raw")

display(deliveries_df)

###  Step 2 - Check Total Records


In [0]:
print("Total Records :", deliveries_df.count())

### ## Step 3 - Duplicate Check (Composite Key)

In [0]:
deliveries_df.groupBy(
    "match_id",
    "innings",
    "over",
    "ball"
).count().filter(col("count") > 1).display()

### Step 4 - Remove Duplicate Deliveries

In [0]:
deliveries_clean_df = deliveries_df.dropDuplicates(
    ["match_id","innings","over","ball"]
)

### Step 5 - Null Value Validation

In [0]:
print("match_id nulls :", deliveries_clean_df.filter(col("match_id").isNull()).count())

print("batsman nulls :", deliveries_clean_df.filter(col("batsman").isNull()).count())

print("bowler nulls :", deliveries_clean_df.filter(col("bowler").isNull()).count())

print("runs_off_bat nulls :", deliveries_clean_df.filter(col("runs_off_bat").isNull()).count())

### Step 6 - Data Type Verification

In [0]:
deliveries_clean_df.printSchema()

Business Rule 1
### Over must be between 1 and 20

In [0]:
deliveries_clean_df.filter(
    (col("over") < 1) |
    (col("over") > 20)
).display()

### Remove Invalid Overs

In [0]:
deliveries_clean_df = deliveries_clean_df.filter(
    (col("over") >= 1) &
    (col("over") <= 20)
)

###Business Rule 2


In [0]:
deliveries_clean_df.filter(
    col("runs_off_bat") < 0
).display()

### Handle Negative Runs

In [0]:
from pyspark.sql.functions import when

deliveries_clean_df = deliveries_clean_df.withColumn(
    "runs_off_bat",
    when(col("runs_off_bat") < 0, 0)
    .otherwise(col("runs_off_bat"))
)

### Handle NULL Values (Future Proof)

In [0]:
deliveries_clean_df = deliveries_clean_df.fillna({
    "batsman": "Unknown",
    "bowler": "Unknown",
    "runs_off_bat": 0,
    "extras": 0
})

### Business Rule 3

In [0]:
deliveries_clean_df.filter(
    trim(col("batsman")) == ""
).show()

deliveries_clean_df.filter(
    trim(col("bowler")) == ""
).show()

### Business Rule 4

In [0]:
deliveries_clean_df.filter(
    col("extras") < 0
).show()

In [0]:
deliveries_clean_df = (
    deliveries_clean_df
    .withColumn("batsman", initcap(trim(col("batsman"))))
    .withColumn("bowler", initcap(trim(col("bowler"))))
)

In [0]:
(deliveries_clean_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("ipl_analytics.silver.deliveries"))

In [0]:
display(
    spark.table("ipl_analytics.silver.deliveries")
)

In [0]:
%sql

SELECT COUNT(*)
FROM ipl_analytics.silver.deliveries;

[](url)